# Vector Quantization in Qdrant: Reproducible Measurements

This notebook reruns the core measurements behind the article [Vector Quantization in Qdrant: Which Method to Use and What You Trade](https://qdrant.tech/articles/vector-quantization-guide/) on one of its datasets, SciFact with Cohere embed-english-v3.0 vectors. Every snippet in the article appears here and runs as written.

What it does:

1. Downloads the pinned SciFact corpus, test queries, and relevance judgments with their published Cohere vectors. No embedding API key is needed.
2. Computes exact float32 neighbors outside Qdrant.
3. Builds one collection per quantization method, measures recall@10 against exact search with rescoring off and on, and times the queries.
4. Sweeps `hnsw_ef` and oversampling to find the fastest point that meets a recall target.
5. Compares `turbo4` storage with float32 originals.
6. Computes the sign-imbalance statistic that predicts where binary quantization works.

You need a Qdrant server at version 1.19 or later. A free [Qdrant Cloud](https://cloud.qdrant.io) cluster works; so does `docker run -p 6333:6333 qdrant/qdrant:v1.19.2` on your own machine. The full benchmark, with four datasets, two models, and the memory and cold-start experiments, is in [qdrant-labs/quantization-comparison-bench](https://github.com/qdrant-labs/quantization-comparison-bench).

The numbers here will differ slightly from the article: the article reports 150 seeded held-out queries, three index builds, and server-side timing on a dedicated VM. This notebook uses all 300 official test queries, one build, and client-side timing, which includes your network path.

In [ ]:
! pip -q install "qdrant-client>=1.19" numpy pyarrow huggingface-hub

## Connect to Qdrant

Enter the cluster URL and API key when prompted, or set `QDRANT_URL` and `QDRANT_API_KEY` in the environment. Nothing is written to the notebook.

In [ ]:
import os
from getpass import getpass

from qdrant_client import QdrantClient, models

url = os.environ.get("QDRANT_URL") or input("Qdrant URL (for example https://xyz.cloud.qdrant.io:6333): ").strip()
api_key = os.environ.get("QDRANT_API_KEY")
if api_key is None and url.startswith("https://"):
    api_key = getpass("Qdrant API key: ") or None

client = QdrantClient(url=url, api_key=api_key, timeout=120)
version = client.info().version
print("Qdrant", version)
assert tuple(int(x) for x in version.split(".")[:2]) >= (1, 19), "This notebook needs Qdrant 1.19 or later"

Qdrant 1.19.2


## Download SciFact with its Cohere vectors

The source is [CohereLabs/beir-embed-english-v3](https://huggingface.co/datasets/CohereLabs/beir-embed-english-v3) at the same pinned revision the article uses. SciFact has 5,183 documents and 300 test queries. The vectors are L2-normalized, so cosine similarity is a dot product.

In [ ]:
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

REVISION = "f018922a46a51388348a307ec2fef28019a33026"
SOURCE = "CohereLabs/beir-embed-english-v3"


def parquet(path):
    local = hf_hub_download(SOURCE, path, repo_type="dataset", revision=REVISION)
    return pq.read_table(local).to_pandas()


corpus_df = parquet("scifact/corpus/0000.parquet").sort_values("_id").reset_index(drop=True)
queries_df = parquet("scifact/queries/test.parquet").sort_values("_id").reset_index(drop=True)
qrels_df = parquet("scifact/qrels/test.parquet")


def matrix(frame):
    values = np.stack(frame["emb"].to_numpy()).astype(np.float32)
    return values / np.linalg.norm(values, axis=1, keepdims=True)


corpus = matrix(corpus_df)
queries = matrix(queries_df)
corpus_ids = corpus_df["_id"].astype(str).tolist()
query_ids = queries_df["_id"].astype(str).tolist()
print(corpus.shape, queries.shape, len(qrels_df), "judgments")

scifact/corpus/0000.parquet: reconstructing file:   0%|          |  0.00B / 14.6MB            

scifact/corpus/0000.parquet: downloading bytes:           |  0.00B            

scifact/queries/test.parquet: reconstructing file:   0%|          |  0.00B /  676kB            

scifact/queries/test.parquet: downloading bytes:           |  0.00B            

scifact/qrels/test.parquet: reconstructing file:   0%|          |  0.00B / 7.36kB            

scifact/qrels/test.parquet: downloading bytes:           |  0.00B            

(5183, 1024) (300, 1024) 339 judgments


## Sign imbalance: will binary quantization work on these vectors?

Binary quantization keeps only the sign of each coordinate. The article found that one statistic predicts its recall better than the dimension count: how far each coordinate's share of positive values sits from one half, averaged over coordinates. Below about 0.15 binary is worth testing; above about 0.19 it did not reach 0.95 recall anywhere in the article's grid. SciFact with Cohere scored 0.182.

In [ ]:
sample = corpus[np.random.default_rng(0).choice(len(corpus), min(20000, len(corpus)), replace=False)]
sign_imbalance = np.abs((sample > 0).mean(axis=0) - 0.5).mean()
print(f"sign imbalance: {sign_imbalance:.3f}")

sign imbalance: 0.182


## Exact float32 neighbors, outside Qdrant

ANN recall@10 is the overlap between what a configuration returns and the exact top ten. Computing the reference with NumPy keeps it independent of any index.

In [ ]:
K = 10
scores = queries @ corpus.T
exact_top = np.argsort(-scores, axis=1)[:, :K]
exact_ids = [[corpus_ids[j] for j in row] for row in exact_top]
print("exact top-10 computed for", len(exact_ids), "queries")

exact top-10 computed for 300 queries


## Helpers: build a collection, query it, score it

The first function is the article's collection snippet made parametric. Originals stay in the `cold` tier, the quantized copy is `pinned`, and `max_indexing_threads` is explicit so builds are comparable. The query helper is the article's `top_ids` function.

In [ ]:
import time
import uuid

from qdrant_client.http.exceptions import ResponseHandlingException

DIM = corpus.shape[1]
RUN = uuid.uuid4().hex[:8]


def retry(call, attempts=5):
    """Retry a request on a dropped connection, with a fresh client the second time."""
    global client
    for attempt in range(attempts):
        try:
            return call()
        except ResponseHandlingException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)
            client = QdrantClient(url=url, api_key=api_key, timeout=120)


def create(name, quantization=None, datatype=models.Datatype.FLOAT32):
    collection = f"qguide_{RUN}_{name}"
    retry(lambda: client.delete_collection(collection))
    client.create_collection(
        collection_name=collection,
        vectors_config=models.VectorParams(
            size=DIM,
            distance=models.Distance.COSINE,
            datatype=datatype,
            memory=models.Memory.COLD,  # originals stay on disk until read
        ),
        quantization_config=quantization,
        hnsw_config=models.HnswConfigDiff(m=16, ef_construct=100, max_indexing_threads=8, full_scan_threshold=10),
        optimizers_config=models.OptimizersConfigDiff(default_segment_number=1, indexing_threshold=1),
    )
    retry(lambda: client.upload_collection(collection_name=collection, vectors=corpus, ids=list(range(len(corpus))), batch_size=512, wait=True))
    for _ in range(600):
        info = retry(lambda: client.get_collection(collection))
        if info.status == models.CollectionStatus.GREEN and info.indexed_vectors_count >= len(corpus):
            break
        time.sleep(1)
    else:
        raise TimeoutError(f"{collection} did not finish indexing")
    return collection


def top_ids(collection, query, **params):
    return [
        point.id
        for point in retry(
            lambda: client.query_points(
                collection_name=collection,
                query=query,
                limit=10,
                search_params=models.SearchParams(**params),
            )
        ).points
    ]


def evaluate(collection, hnsw_ef=256, rescore=None, oversampling=None, warmup=20):
    """Mean recall@10 against exact search and client-side latency in ms."""
    params = {"hnsw_ef": hnsw_ef}
    if rescore is not None:
        params["quantization"] = models.QuantizationSearchParams(rescore=rescore, oversampling=oversampling)
    for i in range(min(warmup, len(queries))):
        top_ids(collection, queries[i].tolist(), **params)
    recalls, latencies = [], []
    for i, query in enumerate(queries):
        started = time.perf_counter()
        ids = top_ids(collection, query.tolist(), **params)
        latencies.append((time.perf_counter() - started) * 1000)
        recalls.append(len({corpus_ids[j] for j in ids} & set(exact_ids[i])) / K)
    return float(np.mean(recalls)), float(np.percentile(latencies, 50)), float(np.percentile(latencies, 95))

## Rescoring is most of the quality

One collection per method, all at `hnsw_ef=256` and oversampling 2, queried with rescoring off and then on. Rescoring is a per-query setting, so the same collection serves both.

In [ ]:
def pinned(config):
    return config


METHODS = {
    "float32": None,
    "scalar8": models.ScalarQuantization(scalar=models.ScalarQuantizationConfig(type=models.ScalarType.INT8, quantile=0.99, memory=models.Memory.PINNED)),
    "binary1": models.BinaryQuantization(binary=models.BinaryQuantizationConfig(encoding=models.BinaryQuantizationEncoding.ONE_BIT, query_encoding=models.BinaryQuantizationQueryEncoding.BINARY, memory=models.Memory.PINNED)),
    "binary2": models.BinaryQuantization(binary=models.BinaryQuantizationConfig(encoding=models.BinaryQuantizationEncoding.TWO_BITS, memory=models.Memory.PINNED)),
    "product8": models.ProductQuantization(product=models.ProductQuantizationConfig(compression=models.CompressionRatio.X8, memory=models.Memory.PINNED)),
    "turbo4": models.TurboQuantization(turbo=models.TurboQuantQuantizationConfig(bits=models.TurboQuantBitSize.BITS4, memory=models.Memory.PINNED)),
    "turbo2": models.TurboQuantization(turbo=models.TurboQuantQuantizationConfig(bits=models.TurboQuantBitSize.BITS2, memory=models.Memory.PINNED)),
    "turbo1": models.TurboQuantization(turbo=models.TurboQuantQuantizationConfig(bits=models.TurboQuantBitSize.BITS1, memory=models.Memory.PINNED)),
}

collections = {}
print(f"{'method':9s} {'rescore off':>22s} {'rescore on':>22s}")
for name, quantization in METHODS.items():
    collections[name] = create(name, quantization)
    if quantization is None:
        recall, p50, p95 = evaluate(collections[name])
        print(f"{name:9s} {recall:6.3f} {p50:5.2f}/{p95:5.2f} ms  (no quantized copy)")
        continue
    off = evaluate(collections[name], rescore=False, oversampling=2.0)
    on = evaluate(collections[name], rescore=True, oversampling=2.0)
    print(f"{name:9s} {off[0]:6.3f} {off[1]:5.2f}/{off[2]:5.2f} ms  {on[0]:6.3f} {on[1]:5.2f}/{on[2]:5.2f} ms")

method               rescore off             rescore on


float32    1.000  0.90/ 1.14 ms  (no quantized copy)


scalar8    0.983  0.74/ 1.07 ms   1.000  1.13/ 2.13 ms


binary1    0.633  0.68/ 0.99 ms   0.805  1.00/ 1.72 ms


binary2    0.719  0.70/ 0.92 ms   0.894  1.10/ 1.68 ms


product8   0.967  1.35/ 1.47 ms   1.000  1.81/ 3.32 ms


turbo4     0.964  0.81/ 1.53 ms   1.000  1.06/ 1.74 ms


turbo2     0.893  0.74/ 0.84 ms   0.996  1.05/ 1.73 ms


turbo1     0.781  0.75/ 0.99 ms   0.949  1.08/ 1.79 ms


Latencies are client-side p50/p95 and include the network between you and the cluster. Recall is the article's definition: overlap with the exact top ten.

## Tune to a target, then compare

The article's rule: for each method, take the fastest point in the grid with recall@10 of at least 0.95. The article does this on 50 development queries and then evaluates frozen points on held-out queries; here, to keep the notebook short, the sweep runs on the test queries directly, so treat it as an illustration of the procedure rather than a held-out result.

In [ ]:
TARGET = 0.95
for name in ("scalar8", "turbo4", "binary1", "product8"):
    best = None
    for hnsw_ef in (64, 128, 256):
        for oversampling in (1.0, 2.0, 4.0, 8.0):
            recall, p50, p95 = evaluate(collections[name], hnsw_ef=hnsw_ef, rescore=True, oversampling=oversampling, warmup=5)
            if recall >= TARGET and (best is None or p95 < best[2]):
                best = (hnsw_ef, oversampling, p95, recall)
    if best:
        print(f"{name:9s} fastest point meeting {TARGET}: hnsw_ef={best[0]}, oversampling={best[1]:g}, p95={best[2]:.2f} ms, recall={best[3]:.3f}")
    else:
        print(f"{name:9s} no point in the grid reached {TARGET}")

scalar8   fastest point meeting 0.95: hnsw_ef=128, oversampling=2, p95=0.73 ms, recall=0.999


turbo4    fastest point meeting 0.95: hnsw_ef=64, oversampling=1, p95=0.73 ms, recall=0.962


binary1   fastest point meeting 0.95: hnsw_ef=128, oversampling=8, p95=0.74 ms, recall=0.968


product8  fastest point meeting 0.95: hnsw_ef=64, oversampling=2, p95=1.12 ms, recall=0.994


## `turbo4` storage versus float32 originals

A storage datatype changes the originals themselves. Three collections: float32 originals with a 1-bit TurboQuant copy, `turbo4` storage with no copy, and `turbo4` storage with the same 1-bit copy. The last one rescores against 4-bit values, so it cannot recover what the datatype discarded.

In [ ]:
turbo1 = models.TurboQuantization(turbo=models.TurboQuantQuantizationConfig(bits=models.TurboQuantBitSize.BITS1, memory=models.Memory.PINNED))
datatype_runs = {
    "float32 + 1-bit copy": create("f32_turbo1", turbo1),
    "turbo4 storage only": create("dt_turbo4", None, datatype=models.Datatype.TURBO4),
    "turbo4 storage + 1-bit copy": create("dt_turbo4_turbo1", turbo1, datatype=models.Datatype.TURBO4),
}
for label, collection in datatype_runs.items():
    has_copy = "copy" in label
    recall, p50, p95 = evaluate(collection, rescore=True if has_copy else None, oversampling=2.0 if has_copy else None)
    print(f"{label:28s} recall@10={recall:.3f}  p95={p95:.2f} ms")

float32 + 1-bit copy         recall@10=0.948  p95=1.74 ms


turbo4 storage only          recall@10=0.963  p95=0.82 ms


turbo4 storage + 1-bit copy  recall@10=0.930  p95=0.94 ms


## Clean up

In [ ]:
for collection in list(collections.values()) + list(datatype_runs.values()):
    retry(lambda: client.delete_collection(collection))
print("deleted", len(collections) + len(datatype_runs), "collections")

deleted 11 collections
